# Install and Import


In [ ]:
!pip install -U -q huggingface_hub[hf_xet]==0.31.2 datasets codecarbon peft bitsandbytes evaluate transformers sentencepiece

In [ ]:
import pandas as pd
import numpy as np
import re
import gc
from datetime import date
from tqdm import tqdm
from codecarbon import EmissionsTracker

import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import datasets
from datasets import load_dataset, Dataset

import peft
from peft import LoraConfig, PeftConfig, get_peft_model, get_peft_config, PeftModel, TaskType, PeftModelForSequenceClassification
from peft import prepare_model_for_kbit_training
import torch
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch import nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight

from huggingface_hub import login
from google.colab import userdata
from google.colab import drive

HF_TOKEN = userdata.get('HF_TOKEN')
HF_READ_TOKEN = userdata.get('HF_READ_TOKEN')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

!nvidia-smi

In [ ]:
# mount to google drive and folder path definition
# drive.mount("/content/drive")
# PATH = ...
# PATH_model = ...
# PATH_data = ...

def drive_refresh():
  drive.flush_and_unmount()
  drive.mount("/content/drive")

def empty_cache():
  gc.collect()
  torch.cuda.empty_cache()
  !nvidia-smi

In [ ]:
empty_cache()

# Hyperparameters


In [ ]:
model_list = [
        "google/flan-t5-base",
        "./Physionet/Clinical-T5-Base",
        "./google/flan-t5-small",

        "google-bert/bert-base-cased",
        "dmis-lab/biobert-v1.1",
        "emilyalsentzer/Bio_ClinicalBERT",
]

model_lora_list = [
    "google/flan-t5-large",
    "./Physionet/Clinical-T5-Large",
    "google/flan-t5-xl",
]

dataset_name_list = [
    "MT+MU+ISCORE+SD",
]

dataset_list = [
    [
    PATH_data+'DS_ISCORE/MT+MU+ISCORE+SD/',
],
    [
    PATH_data+'DS_ISCORE_RISK_LVL/MT+MU+ISCORE+SD/',
],
]

task_list = [
    "barrier",
    "risklvl",
]

num_labels_list = [
    8,
    4,
]

TASK = 0
DATASET = 0
BATCH_SIZE = 32
MAX_SEQ_LEN = 256
MIN_TOK_FREQ = 1

hyper_params = {
    # 'model_checkpoint': model_checkpoint,
    'task': "I-Score",
    # 'NUM_LABELS': NUM_LABELS,
    'batch_size': BATCH_SIZE,
    'max_seq_length': MAX_SEQ_LEN,
    'min_tok_freq': MIN_TOK_FREQ,
    'learning_rate': 3e-5,
    'num_epochs': 7,
    'seed': 42
}

# experiment.log_parameters(hyper_params)

# Data preparation

In [ ]:
def encode(dataset, model_tokenizer):
  encoded_data = model_tokenizer(dataset['Text'], truncation = True, padding = 'max_length', is_split_into_words = False, max_length = MAX_SEQ_LEN)

  return encoded_data

In [ ]:
def load_datasets(task, dataset, model_num, lora = True):
  # drive_refresh()
  if task == 0:
    ds = load_dataset("csv",
        data_files={
        "train": dataset_list[task][dataset]+'train_barrier.csv',
        "valid": dataset_list[task][dataset]+'valid_barrier.csv',
        "test": dataset_list[task][dataset]+'test_barrier.csv',
        },
        keep_default_na = False,
        )
  else:
    ds = load_dataset("csv",
        data_files={
        "train": dataset_list[task][dataset]+'train_risklvl.csv',
        "valid": dataset_list[task][dataset]+'valid_risklvl.csv',
        "test": dataset_list[task][dataset]+'test_risklvl.csv',
        },
        keep_default_na = False,
        )

  ds = ds.cast_column("label", datasets.Value("int64"))
  df_train = pd.DataFrame(ds['train'])
  class_weights = compute_class_weight('balanced', classes=np.unique(df_train['label']), y=df_train['label'])
  class_weights = torch.tensor(class_weights, dtype=torch.float32)

  ds_train = ds['train']
  ds_valid = ds['valid']
  ds_test = ds['test']

  if lora:
    print("tokenizer model = " + model_lora_list[model_num])
    model_tokenizer = AutoTokenizer.from_pretrained(model_lora_list[model_num], token=HF_READ_TOKEN)
  else:
    print("tokenizer model = " + model_list[model_num])
    model_tokenizer = AutoTokenizer.from_pretrained(model_list[model_num], token=HF_READ_TOKEN)

  ds_train_encoded = ds_train.with_format('torch').map(lambda example: encode(example, model_tokenizer), batched = True)
  ds_valid_encoded = ds_valid.with_format('torch').map(lambda example: encode(example, model_tokenizer), batched = True)
  ds_test_encoded = ds_test.with_format('torch').map(lambda example: encode(example, model_tokenizer), batched = True)

  ds_train_loader = DataLoader(ds_train_encoded, batch_size=BATCH_SIZE, shuffle=True)
  ds_valid_loader = DataLoader(ds_valid_encoded, batch_size=BATCH_SIZE, shuffle=True)
  ds_test_loader = DataLoader(ds_test_encoded, batch_size=BATCH_SIZE, shuffle=True)

  return ds_train_encoded, ds_valid_encoded, ds_test_encoded, ds_train_loader, ds_valid_loader, ds_test_loader, class_weights


#Evaluate function

In [ ]:
def compute_metrics(eval_pred):
  predictions, labels = eval_pred
  # predictions = np.argmax(predictions[0], axis = 1) # for Flan T5 series
  predictions = np.argmax(predictions, axis = 1)

  precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)
  accuracy = accuracy_score(labels, predictions)

  return {"accuracy": accuracy, "f1": f1, "precision": precision, "recall": recall}

# Training

## CustomLossTrainer


In [ ]:
class CustomTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(kwargs["model"].device)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

## Normal Trainer

In [ ]:
def train(task_num, dataset_num, model_num, tds, vds, testds, class_weights_tensor):
  task = task_list[task_num]
  num_labels = num_labels_list[task_num]
  dataset = dataset_name_list[dataset_num]
  model_checkpoint = model_list[model_num]

  tds = tds
  vds = vds
  testds = testds

  pretrained_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels = num_labels, token=HF_READ_TOKEN)

  for param in pretrained_model.parameters(): param.data = param.data.contiguous()

  training_args = TrainingArguments(
      output_dir = f"{PATH_model}{model_checkpoint}-{today}finetuned-{task}-{dataset}",
      overwrite_output_dir = True,
      report_to = "wandb",
      eval_strategy = 'epoch',
      save_strategy = 'epoch',
      num_train_epochs = hyper_params['num_epochs'],
      save_strategy = 'best',
      metric_for_best_model = "eval_f1",
      learning_rate = hyper_params['learning_rate'],
      per_device_train_batch_size = hyper_params['batch_size'],
      per_device_eval_batch_size = hyper_params['batch_size'],
      load_best_model_at_end = True,
      seed = hyper_params['seed'],
      warmup_ratio = 0.05,
      lr_scheduler_type = 'cosine_with_restarts',

  )

  trainer = CustomTrainer(
    model=pretrained_model,
    args=training_args,
    train_dataset=tds,
    eval_dataset=vds,
    compute_metrics=compute_metrics,
    callbacks = [transformers.EarlyStoppingCallback(early_stopping_patience=5)],
    class_weights=class_weights_tensor
  )

  trainer.model.config.problem_type = "single_label_classification"

  trainer.train()

  trainer.save_model(f"{PATH_model}{model_checkpoint}-finetuned-{task}-{dataset}")

  test_results = trainer.evaluate(eval_dataset=testds)
  return_string = f"{model_checkpoint}-finetuned-{task}-{dataset}" + "\n" + "eval_accuracy: " + str(test_results['eval_accuracy']) + "\n"
  return_string += "eval_precision: " + str(test_results['eval_precision']) + "\n"
  return_string += "eval_recall: " + str(test_results['eval_recall']) + "\n"
  return_string += "eval_f1: " + str(test_results['eval_f1']) + "\n"
  print(return_string)

  return test_results['eval_accuracy'], test_results['eval_precision'], test_results['eval_recall'], test_results['eval_f1']



## Lora Trainer

In [ ]:
def lora_train(task_num, dataset_num, model_num, tds, vds, testds, class_weights_tensor):
  task = task_list[task_num]
  num_labels = num_labels_list[task_num]
  dataset = dataset_name_list[dataset_num]
  model_checkpoint = model_lora_list[model_num]

  tds = tds
  vds = vds
  testds = testds

  lora_config = LoraConfig(
      task_type = "SEQ_CLS",
      r=16,
      lora_alpha=32,
      target_modules=["q", "v"],
      lora_dropout=0.1,
      bias= 'none',
  )

  base_model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels = num_labels, token=HF_READ_TOKEN)
  base_model.save_pretrained(f"{PATH_model}{model_checkpoint}/{task}")
  peft_model = get_peft_model(base_model, lora_config)
  print(peft_model.print_trainable_parameters())

  training_args = TrainingArguments(
      output_dir = f"{PATH_model}{model_checkpoint}-{today}finetuned-{task}-{dataset}",
      overwrite_output_dir = True,
      eval_strategy = 'steps',
      eval_steps = 500,
      save_strategy = 'best',
      save_steps = 500,
      save_total_limit = 5,
      max_steps = 10000,
      learning_rate = 4e-4,
      auto_find_batch_size = True,
      load_best_model_at_end = True,
      warmup_ratio = 0.05,
      lr_scheduler_type = 'cosine_with_restarts',
      metric_for_best_model = "eval_f1",
      report_to = "wandb",
      seed = hyper_params['seed'],
      )

  lora_trainer = CustomTrainer(
      model = peft_model,
      args = training_args,
      train_dataset = tds,
      eval_dataset = vds,
      compute_metrics = compute_metrics,
      callbacks = [transformers.EarlyStoppingCallback(early_stopping_patience=10)],
      class_weights = class_weights_tensor,
  )

  lora_trainer.train()

  peft_model.save_pretrained(f"{PATH_model}{model_checkpoint}-finetuned-{task}-{dataset}")

  print("model_saved successfully!")

  test_results = lora_trainer.evaluate(eval_dataset=testds)
  return_string = f"{model_checkpoint}-finetuned-{task}-{dataset}" + "\n" + "eval_accuracy: " + str(test_results['eval_accuracy']) + "\n"
  return_string += "eval_f1: " + str(test_results['eval_f1']) + "\n"
  return_string += "eval_precision: " + str(test_results['eval_precision']) + "\n"
  return_string += "eval_recall: " + str(test_results['eval_recall']) + "\n"

  print(return_string)

  return test_results['eval_accuracy'], test_results['eval_precision'], test_results['eval_recall'], test_results['eval_f1']

## Loop_training

In [ ]:
import wandb

def normal_loop_training():
  results = pd.DataFrame(columns=['task', 'dataset', 'model', 'eval_acc', 'eval_prec', 'eval_recall', 'eval_f1', 'emissions'])
  for model_num in range(len(model_list)):
    for task_num in range(len(task_list)):
      for dataset_num in range(len(dataset_name_list)):
        wandb.init(project="huggingface", name=f"{model_list[model_num]}-finetuned-{task_list[task_num]}-{dataset_name_list[dataset_num]}")
        tds, vds, testds, tdl, vdl, testdl, class_weights_tensor = load_datasets(task_num, dataset_num, model_num, lora = False)

        tracker = EmissionsTracker()
        tracker.start()

        eval_acc, eval_prec, eval_recall, eval_f1 = train(task_num, dataset_num, model_num, tds, vds, testds, class_weights_tensor)

        emissions: float = tracker.stop()
        print(f"Emissions: {emissions} kg")

        results = pd.concat([results, pd.DataFrame([{'task': task_list[task_num], 'dataset': dataset_name_list[dataset_num], 'model': model_list[model_num], 'eval_acc':eval_acc, 'eval_prec': eval_prec, 'eval_recall': eval_recall, 'eval_f1':eval_f1, 'emissions': emissions}])], ignore_index=True)

        wandb.finish()
  return results

def lora_loop_training():
  results = pd.DataFrame(columns=['task', 'dataset', 'model', 'eval_acc', 'eval_prec' ,'eval_recall', 'eval_f1', 'emissions'])
  for dataset_num in range(len(dataset_name_list)):
    for model_num in range(len(model_lora_list)):
      for task_num in range(len(task_list)):
        wandb.init(project="huggingface", name=f"{model_lora_list[model_num]}-finetuned-{task_list[task_num]}-{dataset_name_list[dataset_num]}")
        tds, vds, testds, tdl, vdl, testdl, class_weights_tensor = load_datasets(task_num, dataset_num, model_num)

        tracker = EmissionsTracker()
        tracker.start()

        eval_acc, eval_prec, eval_recall, eval_f1 = lora_train(task_num, dataset_num, model_num, tds, vds, testds, class_weights_tensor)

        emissions: float = tracker.stop()
        print(f"Emissions: {emissions} kg")

        results = pd.concat([results, pd.DataFrame([{'task': task_list[task_num], 'dataset': dataset_name_list[dataset_num], 'model': model_lora_list[model_num], 'eval_acc':eval_acc, 'eval_prec': eval_prec, 'eval_recall': eval_recall, 'eval_f1':eval_f1, 'emissions':emissions}])], ignore_index=True)
        wandb.finish()
  return results

## Start normal training

In [ ]:
# %cd to the model saving path

results = normal_loop_training()

# Save results to XXX.csv
# results.to_csv(PATH_data + 'XXX.csv', index=False)

## Start lora training

In [ ]:
# %cd to the model saving path

results = lora_loop_training()

# Save results to XXX.csv
# results.to_csv(PATH_data + 'XXX.csv', index=False)

# Evaluation

## Normal test

In [ ]:
model_checkpoint = model_list[0]
task = task_list[0]
tds, vds, testds, tdl, vdl, ds_test_loader, class_weights_tensor = load_datasets(0, 0, 0, lora = False) #(task, dataset, model_num)

# %cd PATH_model_checkpoint
pretrained_model = AutoModelForSequenceClassification.from_pretrained("./checkpoint-2653", local_files_only = True)

In [ ]:
pretrained_model.to(DEVICE)
pretrained_model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
  test_loader = ds_test_loader

  for batch in tqdm(test_loader):
    input_ids = batch['input_ids'].to(DEVICE)
    attention_mask = batch['attention_mask'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    outputs = pretrained_model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1).cpu().numpy()
    all_predictions.extend(predictions)
    all_labels.extend(labels.cpu().numpy())

precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_predictions, average='macro')
accuracy = accuracy_score(all_labels, all_predictions)

print("\n" + f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

print(classification_report(all_labels, all_predictions, zero_division = 0))

## LORA test

In [ ]:
# %cd PATH_model_checkpoint

task = task_list[0]
tds, vds, testds, tdl, vdl, ds_test_loader, class_weights_tensor = load_datasets(0, 0, 0) #(task, dataset, model_num)
lora_config = LoraConfig.from_pretrained("./checkpoint-8500")
classification_model = AutoModelForSequenceClassification.from_pretrained(lora_config.base_model_name_or_path, num_labels=8, local_files_only=True)
peft_model = PeftModel.from_pretrained(classification_model, "./checkpoint-8500")

In [ ]:
peft_model.eval()
peft_model.to(DEVICE)

all_predictions = []
all_labels = []

with torch.no_grad():
  test_loader = vdl

  for batch in tqdm(test_loader):
    input_ids = batch['input_ids'].to(DEVICE)
    attention_mask = batch['attention_mask'].to(DEVICE)
    labels = batch['label'].to(DEVICE)

    outputs = peft_model(input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1).cpu().numpy()
    all_predictions.extend(predictions)
    all_labels.extend(labels.cpu().numpy())

precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_predictions, average='macro')
accuracy = accuracy_score(all_labels, all_predictions)

print("\n" + f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

print(classification_report(all_labels, all_predictions, zero_division = 0))